# YOLO11 Skin Lesion Training on Kaggle

This notebook is designed for the Kaggle setup shown in the screenshot:
- GPU: P100
- Internet: on
- Mounted dataset: `YOLO_ISIC_2018_Clean`
- Extra mounted raw-mask input: `ISIC2018_Task1_Training_GroundTruth` directory or zip

This notebook is **for the `g8_scope + raw-mask aux` experiment only**.
It clones the public GitHub repo, auto-discovers the mounted dataset under `/kaggle/input`, and runs `train_g8_scope_raw_aux.py`.
If the raw-mask input is missing, the notebook raises an error instead of silently falling back to plain `g8_scope`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/solvynchc/graduation_project.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "graduation_project"
PROJECT_DIR = REPO_DIR / "ultralytics"
YOLO_CONFIG_DIR = WORKDIR / ".yolo_config"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
YOLO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR)

print(f"REPO_DIR={REPO_DIR}")
print(f"PROJECT_DIR={PROJECT_DIR}")
print(f"YOLO_CONFIG_DIR={YOLO_CONFIG_DIR}")


In [ ]:
def ensure_packages(packages: list[str]) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


ensure_packages([
    "polars==1.38.1",
    "ultralytics-thop==2.0.18",
])

print("Extra runtime packages are ready.")


In [ ]:
INPUT_DIR = Path("/kaggle/input")


def find_path(base: Path, name: str) -> Path | None:
    direct = base / name
    if direct.exists():
        return direct
    for match in base.rglob(name):
        return match
    return None


if not INPUT_DIR.exists():
    raise FileNotFoundError("/kaggle/input does not exist")

print("Mounted Kaggle inputs:")
for child in sorted(INPUT_DIR.iterdir()):
    print(f" - {child}")

data_root = find_path(INPUT_DIR, "YOLO_Dataset_Ready")
raw_mask_dir = find_path(INPUT_DIR, "ISIC2018_Task1_Training_GroundTruth")
raw_mask_zip = find_path(INPUT_DIR, "ISIC2018_Task1_Training_GroundTruth.zip")

print(f"data_root={data_root}")
print(f"raw_mask_dir={raw_mask_dir}")
print(f"raw_mask_zip={raw_mask_zip}")

if data_root is None:
    raise FileNotFoundError("Could not find YOLO_Dataset_Ready under /kaggle/input")
if raw_mask_dir is None and raw_mask_zip is None:
    raise FileNotFoundError(
        "This notebook is configured for g8_scope + raw-mask aux. Please add the raw mask dataset or zip to Kaggle inputs."
    )


In [ ]:
train_script = PROJECT_DIR / "train_g8_scope_raw_aux.py"
run_name = "yolo11n_seg_dysample_g8_scope_raw_aux_kaggle"
cmd = [
    sys.executable,
    str(train_script),
    "--data-root", str(data_root),
    "--epochs", "50",
    "--imgsz", "640",
    "--batch", "8",
    "--device", "0",
    "--workers", "2",
    "--name", run_name,
]

if raw_mask_dir is not None:
    cmd.extend(["--raw-mask-dir", str(raw_mask_dir)])
if raw_mask_zip is not None:
    cmd.extend([
        "--raw-mask-zip", str(raw_mask_zip),
        "--raw-mask-zip-root", "ISIC2018_Task1_Training_GroundTruth",
    ])

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
env["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR)

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, env=env, check=True)


In [ ]:
runs_dir = PROJECT_DIR / "runs"
best_weights = sorted(runs_dir.rglob("best.pt"))

print(f"runs_dir={runs_dir}")
print("best weight files:")
for path in best_weights:
    print(f" - {path}")

if not best_weights:
    print("No best.pt found yet. Check the training logs above.")
